# Prompt injection defenses: ábrák és táblázatok

Ez a notebook a teljes, egyetlen futásból származó `attack_defense_full_results.csv` alapján készíti el a dolgozat ábráit és táblázatait. Bemenetként megadható közvetlenül a CSV vagy az azt tartalmazó `tables.zip`. Alapértelmezésben a `tables.zip` fájlt keresi a notebook munkakönyvtárában.

A notebook a régebbi, eltérő checkpointból származó `metrics_full_*.csv` és `payoff_full_*.csv` fájlokat nem használja. A payoffokat a legfrissebb példaszintű eredményekből számolja újra. A sikeres bírói ítéleteket az összes támadásra vetíti, ezért a blokkolt támadások sikertelennek számítanak.

A teljes kiértékelt minta 95 támadásból és 198 benign bemenetből áll. Két benign példát a teljes futás előtt kizártak a korábbi időtúllépések után. A game theory részben a `pay_per_execution` támadói költség továbbra is közelítés: a célmodell prompt tokenjeit használja, amelyek a rögzített rendszerpromptot is tartalmazzák.

Szükséges csomagok: `pandas`, `numpy`, `matplotlib`. A kevert Nash-egyensúlyokhoz a `nashpy` is szükséges; enélkül a tiszta egyensúlyokat számolja ki.


## 1. Betöltés, ellenőrzés és számítások

A következő cella csak függvényeket és konfigurációt definiál. A notebook tényleges feldolgozása a későbbi cellában indul.


In [14]:
#!/usr/bin/env python3
"""Generate thesis figures and tables from the full attack–defense results.

The script reads attack_defense_full_results.csv, either directly or from
tables.zip. It does not use the older metrics_full_*.csv or payoff_full_*.csv
files, because those were generated from an earlier, incomplete checkpoint.

Dependencies: pandas, numpy, matplotlib. The optional mixed-equilibrium table
also uses nashpy when it is installed.

Example:
    python generate_tdk_figures.py --input tables.zip --output tdk_figures
"""

from __future__ import annotations

import argparse
import json
import warnings
import zipfile
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

try:
    import nashpy as nash
except ImportError:  # Pure equilibria are still computed without nashpy.
    nash = None


DEFENSE_ORDER = ["D0", "D1", "D2", "D3", "D4"]
DEFENSE_NAMES = {
    "D0": "No defense",
    "D1": "Rule based input filter",
    "D2": "LLM input classifier",
    "D3": "LLM output filter",
    "D4": "Cascade defense",
}
DEFENSE_COLORS = {
    "D0": "#6b7280",
    "D1": "#4c78a8",
    "D2": "#f58518",
    "D3": "#e45756",
    "D4": "#54a24b",
}

# Saved output from notebook 02. K=9 maximizes silhouette in this run; the
# absolute values are modest, so clusters should be described as exploratory.
CLUSTER_K = np.arange(2, 11)
SILHOUETTE = np.array(
    [0.117953, 0.126765, 0.135395, 0.144520, 0.173600,
     0.182482, 0.191617, 0.224668, 0.219369]
)
DAVIES_BOULDIN = np.array(
    [3.437420, 3.161379, 2.880519, 2.755315, 2.447808,
     2.383631, 2.181936, 2.097428, 2.053542]
)

DEFENDER_SCENARIOS = {
    "Balanced": {"security": 0.60, "false_positive": 0.20, "tokens": 0.10, "latency": 0.10},
    "Security heavy": {"security": 0.80, "false_positive": 0.10, "tokens": 0.05, "latency": 0.05},
    "False positive heavy": {"security": 0.45, "false_positive": 0.40, "tokens": 0.075, "latency": 0.075},
    "Cost heavy": {"security": 0.40, "false_positive": 0.15, "tokens": 0.25, "latency": 0.20},
}
ATTACK_COST_WEIGHT = 0.10
TOLERANCE = 1e-9

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 150,
})


def read_results(input_path: Path) -> pd.DataFrame:
    """Read the current per-example results from a CSV or tables.zip."""
    if input_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(input_path) as archive:
            candidates = [
                name for name in archive.namelist()
                if Path(name).name == "attack_defense_full_results.csv"
            ]
            if not candidates:
                raise FileNotFoundError(
                    "attack_defense_full_results.csv was not found in the ZIP."
                )
            # The archive supplied for this project contains one such file.
            member = candidates[0]
            with archive.open(member) as stream:
                return pd.read_csv(stream)
    if input_path.suffix.lower() == ".csv":
        return pd.read_csv(input_path)
    raise ValueError("Input must be a .csv file or tables.zip.")


def bool_series(values: pd.Series) -> pd.Series:
    """Normalize CSV boolean values while preserving missing entries as False."""
    if values.dtype == bool:
        return values.fillna(False)
    return values.astype("string").str.strip().str.lower().isin({"true", "1", "yes"})


def minmax(values: pd.Series | pd.DataFrame):
    low = values.min().min() if isinstance(values, pd.DataFrame) else values.min()
    high = values.max().max() if isinstance(values, pd.DataFrame) else values.max()
    if pd.isna(low) or pd.isna(high):
        raise ValueError("Cannot normalize missing values.")
    if np.isclose(high, low):
        return values * 0.0
    return (values - low) / (high - low)


def validate_and_prepare(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    required = {
        "example_id", "ground_truth", "strategy_id", "strategy_name",
        "benign_group", "defense_id", "effective_defense_decision",
        "judge_success", "operational_total_tokens",
        "operational_llm_calls", "operational_latency_seconds",
        "target_prompt_tokens",
    }
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = raw.copy()
    df["ground_truth"] = df["ground_truth"].astype("string").str.strip().str.lower()
    df["defense_id"] = df["defense_id"].astype("string").str.strip()
    df["effective_defense_decision"] = (
        df["effective_defense_decision"].astype("string").str.strip().str.lower()
    )
    df["judge_success_bool"] = bool_series(df["judge_success"])
    for column in [
        "operational_total_tokens", "operational_llm_calls",
        "operational_latency_seconds", "target_prompt_tokens",
    ]:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)

    if "error" in df.columns:
        errors = df["error"].notna() & df["error"].astype("string").str.strip().ne("")
        if errors.any():
            raise ValueError(
                f"The result file contains {int(errors.sum())} failed rows. "
                "Resolve or explicitly exclude them before producing final figures."
            )
    if df.duplicated(["example_id", "defense_id"]).any():
        raise ValueError("Duplicate example × defense rows were found.")
    if set(df["ground_truth"].dropna()) - {"attack", "benign"}:
        raise ValueError("Unexpected ground_truth labels were found.")
    if set(df["effective_defense_decision"].dropna()) - {"allow", "block"}:
        raise ValueError("Unexpected effective defense decisions were found.")

    defenses = [d for d in DEFENSE_ORDER if d in set(df["defense_id"])]
    if defenses != DEFENSE_ORDER:
        raise ValueError(f"Expected all five defenses {DEFENSE_ORDER}; found {defenses}.")
    if df.groupby("defense_id").size().nunique() != 1:
        raise ValueError("The number of examples differs between defenses.")

    attacks = df.loc[df["ground_truth"].eq("attack")].copy()
    benign = df.loc[df["ground_truth"].eq("benign")].copy()
    if attacks["strategy_id"].isna().any():
        raise ValueError("At least one attack has no strategy_id.")
    passed_attacks = attacks["effective_defense_decision"].eq("allow")
    if attacks.loc[passed_attacks, "judge_success"].isna().any():
        raise ValueError(
            "At least one attack passed to the target has no judge result. "
            "End-to-end attack success cannot be computed safely."
        )
    return df, attacks, benign


def summarize_results(df: pd.DataFrame, attacks: pd.DataFrame, benign: pd.DataFrame):
    defense_rows = []
    for defense in DEFENSE_ORDER:
        all_rows = df[df["defense_id"].eq(defense)]
        a = attacks[attacks["defense_id"].eq(defense)]
        b = benign[benign["defense_id"].eq(defense)]
        blocked_attack = a["effective_defense_decision"].eq("block")
        blocked_benign = b["effective_defense_decision"].eq("block")
        tp = int(blocked_attack.sum())
        fp = int(blocked_benign.sum())
        fn = int((~blocked_attack).sum())
        f1 = (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else np.nan
        defense_rows.append({
            "defense_id": defense,
            "defense": DEFENSE_NAMES[defense],
            "attack_samples": len(a),
            "attack_blocks": tp,
            "attack_block_rate": tp / len(a) if len(a) else np.nan,
            "attack_pass_rate": 1 - tp / len(a) if len(a) else np.nan,
            # Blocked attacks have no target response and count as unsuccessful.
            "end_to_end_judge_successes": int(a["judge_success_bool"].sum()),
            "end_to_end_asr": float(a["judge_success_bool"].sum() / len(a)) if len(a) else np.nan,
            "benign_samples": len(b),
            "false_positives": fp,
            "fpr": fp / len(b) if len(b) else np.nan,
            "f1": f1,
            "mean_operational_tokens": all_rows["operational_total_tokens"].mean(),
            "mean_operational_calls": all_rows["operational_llm_calls"].mean(),
            "mean_operational_latency_seconds": all_rows["operational_latency_seconds"].mean(),
            "p95_operational_latency_seconds": all_rows["operational_latency_seconds"].quantile(0.95),
        })
    defense_summary = pd.DataFrame(defense_rows)

    strategy_rows = []
    for (strategy_id, strategy_name, defense), group in attacks.groupby(
        ["strategy_id", "strategy_name", "defense_id"], dropna=False, sort=True
    ):
        strategy_rows.append({
            "strategy_id": strategy_id,
            "strategy_name": strategy_name,
            "defense_id": defense,
            "attack_samples": group["example_id"].nunique(),
            "attack_block_rate": group["effective_defense_decision"].eq("block").mean(),
            "end_to_end_asr": group["judge_success_bool"].sum() / group["example_id"].nunique(),
            "mean_operational_tokens": group["operational_total_tokens"].mean(),
            "mean_target_prompt_tokens": group["target_prompt_tokens"].mean(),
        })
    strategy_summary = pd.DataFrame(strategy_rows)

    benign_rows = []
    for (group_name, defense), group in benign.groupby(["benign_group", "defense_id"], sort=True):
        benign_rows.append({
            "benign_group": group_name,
            "defense_id": defense,
            "samples": group["example_id"].nunique(),
            "false_positives": int(group["effective_defense_decision"].eq("block").sum()),
            "false_positive_rate": group["effective_defense_decision"].eq("block").mean(),
        })
    benign_summary = pd.DataFrame(benign_rows)
    return defense_summary, strategy_summary, benign_summary


def make_payoffs(
    df: pd.DataFrame,
    attacks: pd.DataFrame,
    defense_summary: pd.DataFrame,
    strategy_summary: pd.DataFrame,
):
    strategies = sorted(strategy_summary["strategy_id"].dropna().unique())
    names = (
        strategy_summary[["strategy_id", "strategy_name"]]
        .drop_duplicates().set_index("strategy_id")["strategy_name"].to_dict()
    )
    block = strategy_summary.pivot(
        index="strategy_id", columns="defense_id", values="attack_block_rate"
    ).reindex(index=strategies, columns=DEFENSE_ORDER)
    asr = strategy_summary.pivot(
        index="strategy_id", columns="defense_id", values="end_to_end_asr"
    ).reindex(index=strategies, columns=DEFENSE_ORDER)
    token_cost = strategy_summary.pivot(
        index="strategy_id", columns="defense_id", values="mean_operational_tokens"
    ).reindex(index=strategies, columns=DEFENSE_ORDER)
    normalized_tokens = minmax(token_cost)

    lookup = defense_summary.set_index("defense_id").reindex(DEFENSE_ORDER)
    fpr = lookup["fpr"].astype(float)
    latency = minmax(lookup["mean_operational_latency_seconds"].astype(float))
    fpr_matrix = pd.DataFrame(
        np.tile(fpr.to_numpy(), (len(strategies), 1)), index=strategies, columns=DEFENSE_ORDER
    )
    latency_matrix = pd.DataFrame(
        np.tile(latency.to_numpy(), (len(strategies), 1)), index=strategies, columns=DEFENSE_ORDER
    )

    # This reproduces the documented execution-token proxy, averaged by
    # strategy across defenses. It includes the fixed target system prompt;
    # interpret the pay-per-execution case as a sensitivity scenario only.
    execution_by_cell = strategy_summary.pivot(
        index="strategy_id", columns="defense_id", values="mean_target_prompt_tokens"
    ).reindex(index=strategies, columns=DEFENSE_ORDER)
    attack_cost = minmax(execution_by_cell.mean(axis=1))

    equilibrium_rows = []
    mixed_rows = []
    scenario_summary = []
    base_attacker = base_defender = None
    base_pure = []

    for scenario_name, weights in DEFENDER_SCENARIOS.items():
        defender = (
            weights["security"] * block
            - weights["false_positive"] * fpr_matrix
            - weights["tokens"] * normalized_tokens
            - weights["latency"] * latency_matrix
        )
        for cost_mode in ["free_prompt_creation", "pay_per_execution"]:
            costs = 0.0 if cost_mode == "free_prompt_creation" else ATTACK_COST_WEIGHT * attack_cost
            attacker = asr.subtract(costs, axis=0)
            a = attacker.to_numpy(dtype=float)
            d = defender.to_numpy(dtype=float)
            attacker_best = a >= a.max(axis=0, keepdims=True) - TOLERANCE
            defender_best = d >= d.max(axis=1, keepdims=True) - TOLERANCE
            pure_cells = [tuple(x) for x in np.argwhere(attacker_best & defender_best)]

            for ai, di in pure_cells:
                equilibrium_rows.append({
                    "defender_scenario": scenario_name,
                    "attack_cost_mode": cost_mode,
                    "strategy_id": strategies[ai],
                    "strategy_name": names[strategies[ai]],
                    "defense_id": DEFENSE_ORDER[di],
                    "attacker_payoff": a[ai, di],
                    "defender_payoff": d[ai, di],
                })

            mixed_count = 0
            if nash is not None and a.shape[0] > 1:
                game = nash.Game(a, d)
                seen = set()
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    for ap, dp in game.support_enumeration(tol=TOLERANCE):
                        ap = np.asarray(ap, dtype=float)
                        dp = np.asarray(dp, dtype=float)
                        if not np.isfinite(ap).all() or not np.isfinite(dp).all():
                            continue
                        if np.count_nonzero(ap > TOLERANCE) == 1 and np.count_nonzero(dp > TOLERANCE) == 1:
                            continue
                        signature = (tuple(np.round(ap, 9)), tuple(np.round(dp, 9)))
                        if signature in seen:
                            continue
                        seen.add(signature)
                        mixed_count += 1
                        mixed_rows.append({
                            "defender_scenario": scenario_name,
                            "attack_cost_mode": cost_mode,
                            "attacker_distribution": json.dumps(
                                {names[strategies[i]]: float(p) for i, p in enumerate(ap) if p > TOLERANCE},
                                ensure_ascii=False,
                            ),
                            "defender_distribution": json.dumps(
                                {DEFENSE_ORDER[i]: float(p) for i, p in enumerate(dp) if p > TOLERANCE}
                            ),
                            "attacker_expected_payoff": float(ap @ a @ dp),
                            "defender_expected_payoff": float(ap @ d @ dp),
                        })

            scenario_summary.append({
                "defender_scenario": scenario_name,
                "attack_cost_mode": cost_mode,
                "pure_equilibrium_count": len(pure_cells),
                "pure_equilibria": "; ".join(
                    f"{strategies[ai]} × {DEFENSE_ORDER[di]}" for ai, di in pure_cells
                ),
                "mixed_equilibrium_count": mixed_count if nash is not None else np.nan,
            })
            if scenario_name == "Balanced" and cost_mode == "free_prompt_creation":
                base_attacker, base_defender, base_pure = attacker, defender, pure_cells

    return (
        strategies, names, block, asr, base_attacker, base_defender, base_pure,
        pd.DataFrame(equilibrium_rows), pd.DataFrame(mixed_rows), pd.DataFrame(scenario_summary),
    )


def heatmap(ax, matrix: pd.DataFrame, title: str, cmap: str, fmt: str = ".2f", stars=None):
    values = matrix.to_numpy(dtype=float)
    image = ax.imshow(values, cmap=cmap, vmin=0 if values.min() >= 0 else None,
                      vmax=1 if values.min() >= 0 and values.max() <= 1 else None, aspect="auto")
    ax.set_xticks(np.arange(len(matrix.columns)), matrix.columns)
    ax.set_yticks(np.arange(len(matrix.index)), matrix.index)
    ax.set_xlabel("Defense")
    ax.set_ylabel("Attack strategy")
    ax.set_title(title)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            marker = " ★" if stars and (i, j) in stars else ""
            ax.text(j, i, f"{values[i, j]:{fmt}}{marker}", ha="center", va="center", fontsize=8)
    return image


def build_figures(
    output_dir: Path,
    df: pd.DataFrame,
    attacks: pd.DataFrame,
    benign: pd.DataFrame,
    defense_summary: pd.DataFrame,
    strategy_summary: pd.DataFrame,
    benign_summary: pd.DataFrame,
    game_data,
):
    figure_dir = output_dir / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    figures: list[tuple[str, plt.Figure]] = []

    def keep(name: str, fig: plt.Figure):
        fig.tight_layout()
        figures.append((name, fig))
        fig.savefig(figure_dir / f"{name}.pdf", bbox_inches="tight")
        fig.savefig(figure_dir / f"{name}.png", dpi=300, bbox_inches="tight")

    # 1. Research and analysis pipeline.
    fig, ax = plt.subplots(figsize=(8.2, 10.2))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    stages = [
        "IPI Arena attack prompts\n95 examples, 28 behavior IDs",
        "Text embeddings\nall-mpnet-base-v2, 768 dimensions",
        "Exploratory clustering\nKMeans, k = 2…10; silhouette and Davies–Bouldin scores",
        "LLM assisted interpretation\n9 clusters consolidated into 3 attack strategy groups",
        "Benign control set\n200 selected; 2 examples excluded after timeouts",
        "Five defense configurations\nD0–D4, including input, output, and cascade filters",
        "Full experiment\n293 inputs × 5 defenses = 1,465 completed pairs",
        "Evaluation and game model\nclassification, judge outcomes, operational costs, payoff sensitivity",
    ]
    y_positions = np.linspace(0.91, 0.09, len(stages))
    for idx, (label, y) in enumerate(zip(stages, y_positions)):
        box = FancyBboxPatch((0.12, y - 0.041), 0.76, 0.082,
                             boxstyle="round,pad=0.012", facecolor="#f3f6fa",
                             edgecolor="#59718c", linewidth=1.0)
        ax.add_patch(box)
        ax.text(0.5, y, label, ha="center", va="center", fontsize=9)
        if idx < len(stages) - 1:
            ax.add_patch(FancyArrowPatch((0.5, y - 0.045),
                                         (0.5, y_positions[idx + 1] + 0.045),
                                         arrowstyle="-|>", mutation_scale=11,
                                         color="#59718c", linewidth=1.0))
    ax.set_title("Research workflow", pad=12)
    keep("01_workflow", fig)

    # 2. Clustering diagnostics from notebook 02.
    fig, ax1 = plt.subplots(figsize=(8.0, 4.8))
    ax1.plot(CLUSTER_K, SILHOUETTE, marker="o", color="#4c78a8", label="Silhouette score")
    ax1.set_xlabel("Number of clusters, k")
    ax1.set_ylabel("Silhouette score", color="#4c78a8")
    ax1.tick_params(axis="y", labelcolor="#4c78a8")
    ax1.set_xticks(CLUSTER_K)
    ax1.axvline(9, color="#4c78a8", linestyle="--", alpha=0.6)
    ax2 = ax1.twinx()
    ax2.plot(CLUSTER_K, DAVIES_BOULDIN, marker="s", color="#f58518", label="Davies–Bouldin index")
    ax2.set_ylabel("Davies–Bouldin index", color="#f58518")
    ax2.tick_params(axis="y", labelcolor="#f58518")
    ax1.set_title("Clustering quality across candidate values of k")
    ax1.text(0.01, -0.22, "k = 9 maximizes silhouette; the scores indicate exploratory, overlapping groups.",
             transform=ax1.transAxes, fontsize=8)
    keep("02_cluster_diagnostics", fig)

    # 3. Attack strategy distribution.
    counts = attacks.drop_duplicates("example_id").groupby(
        ["strategy_id", "strategy_name"], dropna=False
    )["example_id"].nunique().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8.0, 4.5))
    labels = [f"{sid}: {name}" for sid, name in counts.index]
    bars = ax.barh(labels, counts.values, color="#4c78a8")
    ax.bar_label(bars, padding=3)
    ax.set_xlabel("Number of attack examples")
    ax.set_title("Attack strategy set and sample imbalance")
    ax.set_xlim(0, max(counts.values) * 1.2)
    keep("03_attack_strategy_counts", fig)

    # 4. Evaluated sample composition.
    unique = df.drop_duplicates("example_id")
    comp = unique.groupby(["ground_truth", "benign_group"], dropna=False).size()
    attack_count = int(unique["ground_truth"].eq("attack").sum())
    general_count = int(((unique["ground_truth"] == "benign") & (unique["benign_group"] == "general")).sum())
    instruction_count = int(((unique["ground_truth"] == "benign") & (unique["benign_group"] == "instruction_heavy")).sum())
    fig, ax = plt.subplots(figsize=(7.8, 4.3))
    labels = ["Attack", "Benign: general", "Benign: instruction heavy"]
    values = [attack_count, general_count, instruction_count]
    bars = ax.bar(labels, values, color=["#e45756", "#72b7b2", "#f2cf5b"])
    ax.bar_label(bars, padding=3)
    ax.set_ylabel("Unique examples in the evaluated run")
    ax.set_title("Composition of the evaluated input set")
    ax.tick_params(axis="x", rotation=12)
    keep("04_sample_composition", fig)

    # 5. Defense detection and end-to-end judge outcome.
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    x = np.arange(len(DEFENSE_ORDER)); width = 0.34
    ordered = defense_summary.set_index("defense_id").reindex(DEFENSE_ORDER)
    ax.bar(x - width / 2, ordered["attack_block_rate"] * 100, width,
           label="Attack block rate", color="#4c78a8")
    ax.bar(x + width / 2, ordered["end_to_end_asr"] * 100, width,
           label="End-to-end judge success rate", color="#e45756")
    ax.set_xticks(x, DEFENSE_ORDER)
    ax.set_ylabel("Percent of all attack inputs")
    ax.set_ylim(0, 100)
    ax.set_title("Attack blocking and observed end-to-end attack success")
    ax.legend(frameon=False, loc="upper right")
    ax.text(0.01, -0.20,
            "Blocked attacks count as unsuccessful. Judge success is generic and does not reproduce the IPI Arena evaluator.",
            transform=ax.transAxes, fontsize=8)
    keep("05_defense_security_outcomes", fig)

    # 6. False positives by benign group.
    pivot_fp = benign_summary.pivot(index="benign_group", columns="defense_id", values="false_positive_rate")
    pivot_fp = pivot_fp.reindex(columns=DEFENSE_ORDER)
    fig, ax = plt.subplots(figsize=(8.0, 4.5))
    x = np.arange(len(pivot_fp.index)); width = 0.15
    for j, defense in enumerate(DEFENSE_ORDER):
        ax.bar(x + (j - 2) * width, pivot_fp[defense].to_numpy() * 100, width,
               label=defense, color=DEFENSE_COLORS[defense])
    ax.set_xticks(x, [str(x).replace("_", " ") for x in pivot_fp.index])
    ax.set_ylabel("False positive rate (%)")
    ax.set_title("False positive rate by benign input group")
    ax.legend(ncol=5, frameon=False)
    keep("06_benign_false_positives", fig)

    # 7. Operational cost and latency.
    fig, ax = plt.subplots(figsize=(7.6, 5.0))
    for _, row in ordered.iterrows():
        d = row.name
        ax.scatter(row["mean_operational_tokens"], row["mean_operational_latency_seconds"],
                   s=95, color=DEFENSE_COLORS[d], label=d, edgecolor="white", linewidth=0.8)
        ax.annotate(d, (row["mean_operational_tokens"], row["mean_operational_latency_seconds"]),
                    xytext=(5, 5), textcoords="offset points", fontsize=9)
    ax.set_xlabel("Mean operational tokens per input")
    ax.set_ylabel("Mean operational latency (seconds)")
    ax.set_title("Security filter operating cost")
    ax.grid(alpha=0.2)
    keep("07_operational_cost", fig)

    # 8. Strategy-level attack blocking and judge success heatmaps.
    block_matrix = strategy_summary.pivot(index="strategy_id", columns="defense_id", values="attack_block_rate").reindex(columns=DEFENSE_ORDER)
    asr_matrix = strategy_summary.pivot(index="strategy_id", columns="defense_id", values="end_to_end_asr").reindex(columns=DEFENSE_ORDER)
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
    im1 = heatmap(axes[0], block_matrix, "Attack block rate", "Blues", ".2f")
    im2 = heatmap(axes[1], asr_matrix, "End-to-end judge success rate", "OrRd", ".2f")
    fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    fig.suptitle("Results by attack strategy and defense", y=1.02)
    keep("08_strategy_defense_heatmaps", fig)

    # 9. Base payoff heatmaps from refreshed raw results, not stale payoff CSVs.
    strategies, names, payoff_block, payoff_asr, attacker, defender, pure, eqs, mixed, sensitivity = game_data
    base_attack_payoff = attacker.to_numpy(dtype=float)
    base_defender_payoff = defender.to_numpy(dtype=float)
    row_labels = [f"{s}: {names[s]}" for s in strategies]
    base_attack_payoff = pd.DataFrame(base_attack_payoff, index=row_labels, columns=DEFENSE_ORDER)
    base_defender_payoff = pd.DataFrame(base_defender_payoff, index=row_labels, columns=DEFENSE_ORDER)
    pure_labeled = [(i, j) for i, j in pure]
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
    im1 = heatmap(axes[0], base_attack_payoff, "Attacker payoff", "YlOrRd", ".2f", pure_labeled)
    im2 = heatmap(axes[1], base_defender_payoff, "Defender payoff", "YlGn", ".2f", pure_labeled)
    fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    fig.suptitle("Balanced weights, free prompt creation; stars mark pure Nash equilibria", y=1.02)
    keep("09_base_payoff_matrices", fig)

    # 10. Pure equilibrium sensitivity under the four weight sets and two cost assumptions.
    sens_pivot = sensitivity.pivot(index="defender_scenario", columns="attack_cost_mode",
                                   values="pure_equilibrium_count")
    sens_pivot = sens_pivot.reindex(index=list(DEFENDER_SCENARIOS.keys()),
                                    columns=["free_prompt_creation", "pay_per_execution"])
    fig, ax = plt.subplots(figsize=(8.0, 4.4))
    image = ax.imshow(sens_pivot.to_numpy(dtype=float), cmap="Purples", aspect="auto")
    ax.set_xticks(np.arange(len(sens_pivot.columns)), ["Free creation", "Pay per execution"])
    ax.set_yticks(np.arange(len(sens_pivot.index)), sens_pivot.index)
    ax.set_title("Number of pure equilibria across payoff assumptions")
    for i in range(sens_pivot.shape[0]):
        for j in range(sens_pivot.shape[1]):
            ax.text(j, i, str(int(sens_pivot.iloc[i, j])), ha="center", va="center")
    fig.colorbar(image, ax=ax, label="Pure equilibrium count")
    ax.text(0.0, -0.19,
            "Pay-per-execution uses target prompt tokens as a proxy; these include a fixed system prompt.",
            transform=ax.transAxes, fontsize=8)
    keep("10_equilibrium_sensitivity", fig)

    with PdfPages(output_dir / "tdk_figures.pdf") as pdf:
        for _, fig in figures:
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)


def write_tables(output_dir: Path, defense_summary: pd.DataFrame,
                 strategy_summary: pd.DataFrame, benign_summary: pd.DataFrame,
                 game_data) -> None:
    table_dir = output_dir / "tables"
    table_dir.mkdir(parents=True, exist_ok=True)
    strategies, names, block, asr, attacker, defender, pure, equilibria, mixed, sensitivity = game_data
    tables = {
        "defense_summary": defense_summary,
        "strategy_by_defense": strategy_summary,
        "benign_false_positives": benign_summary,
        "pure_equilibria": equilibria,
        "mixed_equilibria": mixed,
        "payoff_sensitivity": sensitivity,
    }
    for name, table in tables.items():
        table.to_csv(table_dir / f"{name}.csv", index=False)
        (table_dir / f"{name}.tex").write_text(
            table.to_latex(index=False, escape=True, float_format=lambda x: f"{x:.3f}"),
            encoding="utf-8",
        )


## 2. Bemenet és kimeneti mappa

Ha máshol van az adatfájl, az `INPUT_PATH` értékét írd át a CSV vagy a ZIP teljes elérési útjára. A kimenet a `tdk_figures` mappába kerül.


In [15]:
INPUT_PATH = Path("C:\\work\\LLM filtering\\TDK\\code\\results\\attack_defense_full_results.csv")  # vagy például Path("/path/to/attack_defense_full_results.csv")
OUTPUT_DIR = Path("tdk_figures")

raw_results = read_results(INPUT_PATH)
df, attacks, benign = validate_and_prepare(raw_results)
defense_summary, strategy_summary, benign_summary = summarize_results(df, attacks, benign)
game_data = make_payoffs(df, attacks, defense_summary, strategy_summary)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
defense_summary.to_csv(OUTPUT_DIR / "defense_summary.csv", index=False)
strategy_summary.to_csv(OUTPUT_DIR / "strategy_summary.csv", index=False)
write_tables(OUTPUT_DIR, defense_summary, strategy_summary, benign_summary, game_data)

print(f"Beolvasott sorok: {len(df)}")
print(f"Egyedi bemenetek: {df['example_id'].nunique()} (támadás: {attacks['example_id'].nunique()}, benign: {benign['example_id'].nunique()})")
print("Védekezési összesítés:")
display(defense_summary.round(3))
print("Stratégia × védelem összesítés:")
display(strategy_summary.round(3))


Beolvasott sorok: 1465
Egyedi bemenetek: 293 (támadás: 95, benign: 198)
Védekezési összesítés:


,defense_id,defense,attack_samples,attack_blocks,attack_block_rate,attack_pass_rate,end_to_end_judge_successes,end_to_end_asr,benign_samples,false_positives,fpr,f1,mean_operational_tokens,mean_operational_calls,mean_operational_latency_seconds,p95_operational_latency_seconds
0,D0,No defense,95,0,0.000,1.000,71,0.747,198,0,0.000,0.000,1701.181,1.000,38.448,117.313
1,D1,Rule based input filter,95,2,0.021,0.979,66,0.695,198,0,0.000,0.041,1719.328,0.993,40.813,128.178
2,D2,LLM input classifier,95,13,0.137,0.863,60,0.632,198,0,0.000,0.241,2442.618,1.956,49.726,137.464
3,D3,LLM output filter,95,62,0.653,0.347,14,0.147,198,3,0.015,0.775,2922.952,2.000,51.880,144.973
4,D4,Cascade defense,95,17,0.179,0.821,55,0.579,198,0,0.000,0.304,2390.396,1.935,47.159,134.388


Stratégia × védelem összesítés:


,strategy_id,strategy_name,defense_id,attack_samples,attack_block_rate,end_to_end_asr,mean_operational_tokens,mean_target_prompt_tokens
0,S1,Fake Directive Manipulation,D0,81,0.000,0.741,1879.160,663.630
1,S1,Fake Directive Manipulation,D1,81,0.025,0.704,1721.864,649.753
2,S1,Fake Directive Manipulation,D2,81,0.111,0.654,2975.309,631.420
3,S1,Fake Directive Manipulation,D3,81,0.667,0.148,3364.025,663.630
4,S1,Fake Directive Manipulation,D4,81,0.160,0.593,2828.025,613.691
5,S2,Stealthy Dependency Injection,D0,7,0.000,0.857,1826.143,301.714
6,S2,Stealthy Dependency Injection,D1,7,0.000,0.571,2167.857,301.714
7,S2,Stealthy Dependency Injection,D2,7,0.429,0.429,2690.286,176.286
8,S2,Stealthy Dependency Injection,D3,7,0.571,0.143,3987.571,301.714
9,S2,Stealthy Dependency Injection,D4,7,0.286,0.429,2259.857,216.571


## 3. Az eredmény és a táblázatok áttekintése

Az összesített táblák megjelennek a notebookban, és CSV, illetve LaTeX formátumban is elmentődnek. A LaTeX fájlokat be lehet illeszteni a dolgozatba, a végső feliratokat és változóneveket azonban érdemes a dolgozat terminológiájához igazítani.


In [16]:
defense_summary.round(3)


,defense_id,defense,attack_samples,attack_blocks,attack_block_rate,attack_pass_rate,end_to_end_judge_successes,end_to_end_asr,benign_samples,false_positives,fpr,f1,mean_operational_tokens,mean_operational_calls,mean_operational_latency_seconds,p95_operational_latency_seconds
0,D0,No defense,95,0,0.000,1.000,71,0.747,198,0,0.000,0.000,1701.181,1.000,38.448,117.313
1,D1,Rule based input filter,95,2,0.021,0.979,66,0.695,198,0,0.000,0.041,1719.328,0.993,40.813,128.178
2,D2,LLM input classifier,95,13,0.137,0.863,60,0.632,198,0,0.000,0.241,2442.618,1.956,49.726,137.464
3,D3,LLM output filter,95,62,0.653,0.347,14,0.147,198,3,0.015,0.775,2922.952,2.000,51.880,144.973
4,D4,Cascade defense,95,17,0.179,0.821,55,0.579,198,0,0.000,0.304,2390.396,1.935,47.159,134.388


In [17]:
strategy_summary.round(3)


,strategy_id,strategy_name,defense_id,attack_samples,attack_block_rate,end_to_end_asr,mean_operational_tokens,mean_target_prompt_tokens
0,S1,Fake Directive Manipulation,D0,81,0.000,0.741,1879.160,663.630
1,S1,Fake Directive Manipulation,D1,81,0.025,0.704,1721.864,649.753
2,S1,Fake Directive Manipulation,D2,81,0.111,0.654,2975.309,631.420
3,S1,Fake Directive Manipulation,D3,81,0.667,0.148,3364.025,663.630
4,S1,Fake Directive Manipulation,D4,81,0.160,0.593,2828.025,613.691
5,S2,Stealthy Dependency Injection,D0,7,0.000,0.857,1826.143,301.714
6,S2,Stealthy Dependency Injection,D1,7,0.000,0.571,2167.857,301.714
7,S2,Stealthy Dependency Injection,D2,7,0.429,0.429,2690.286,176.286
8,S2,Stealthy Dependency Injection,D3,7,0.571,0.143,3987.571,301.714
9,S2,Stealthy Dependency Injection,D4,7,0.286,0.429,2259.857,216.571


## 4. Ábrák és PDF

A `tdk_figures.pdf` egy többoldalas, vektoros PDF. A `figures` mappában minden ábra külön PDF és nagy felbontású PNG fájlként is megtalálható. A dolgozatba csak azokat az ábrákat emeld át, amelyek közvetlenül támogatják az adott fejezet érvelését.


In [ ]:
build_figures(OUTPUT_DIR, df, attacks, benign, defense_summary, strategy_summary, benign_summary, game_data)

print(f"Többoldalas PDF: {OUTPUT_DIR / 'tdk_figures.pdf'}")
print(f"Egyedi PDF és PNG ábrák: {OUTPUT_DIR / 'figures'}")
print(f"CSV és LaTeX táblák: {OUTPUT_DIR / 'tables'}")
if nash is None:
    print("A nashpy nincs telepítve: a tiszta egyensúlyok elkészültek, a kevert egyensúlyok táblája üres.")


C:\Users\zalan\AppData\Local\Temp\ipykernel_14104\1460158104.py:399: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) DejaVu Serif.
  fig.tight_layout()
C:\Users\zalan\AppData\Local\Temp\ipykernel_14104\1460158104.py:401: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) DejaVu Serif.
  fig.savefig(figure_dir / f"{name}.pdf", bbox_inches="tight")
C:\Users\zalan\AppData\Local\Temp\ipykernel_14104\1460158104.py:402: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) DejaVu Serif.
  fig.savefig(figure_dir / f"{name}.png", dpi=300, bbox_inches="tight")
C:\Users\zalan\AppData\Local\Temp\ipykernel_14104\1460158104.py:572: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) DejaVu Serif.
  pdf.savefig(fig, bbox_inches="tight")


Többoldalas PDF: tdk_figures\tdk_figures.pdf
Egyedi PDF és PNG ábrák: tdk_figures\figures
CSV és LaTeX táblák: tdk_figures\tables


: 